# Problem 3 — Phase 3: Linear SVM slack `C` (global + optional per-parent-depth)

**Index:** [`hierarchical_problem3_index.ipynb`](hierarchical_problem3_index.ipynb) · **Prev:** [Phase 2](hierarchical_problem3_phase2_truncated_svd.ipynb) · **Next:** [Phase 4](hierarchical_problem3_phase4_kernel_pilot.ipynb)

**Depth:** [`BinaryEdgeSpec.depth`](topic_hierarchy.py) = **parent** distance from `Root` (`depth=0` on Root→H1 edges).

**Slack `C`:** In `LinearSVC`, **`C`** is the regularization strength (inverse slack penalty). The **default is `C=1.0`**. We **do not** refit the full tree at `C=1` here — that run is the same TF-IDF **`LinearSVC`** row as **[`hierarchical_1b.ipynb`](hierarchical_1b.ipynb)** (saved in **`full_tree_rows.pkl`**). The **global grid** below trains **only other `C` values** (e.g. `1e-2`, `0.1`, `10`); the results cell **merges** in the `C=1` row from the pickle.

**Notebook layout**

1. **Setup** — pool + tree; helper to load **`C=1`** metrics from **`full_tree_rows.pkl`**.
2. **Training — global `C`** — optional grid (**no `1.0`** in `C_GRID`).
3. **Results — global `C`** — `C=1` from 1b + trained rows → table + **`problem3_phase3_global_C.csv`**.
4. **Training — per-depth `C` map** — optional; tune **`C_BY_DEPTH`** / **`DEFAULT_C`** (slack for depths not in the map).
5. **Results — per-depth** — table + CSV.

Use **`pooled_edge_f1_stats_by_parent_depth`** when you have a fitted `router` in memory.


In [2]:
from __future__ import annotations
import json
import pickle

import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from hierarchical_classifier import (
    binary_edge_factory,
    linear_svc_estimator_by_depth,
    svm_linear_binary_edge_factory,
)
from hierarchical_summary_metrics import (
    comparison_table,
    pooled_edge_f1_stats_by_parent_depth,
    train_full_tree_and_summarize,
)
from hierarchical_training_data import make_multilabel_binary_pool_data
from topic_hierarchy import load_topic_tree


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found")


def row_linear_svc_c1_from_1b(base: Path) -> dict:
    """
    Default slack C=1 for TF-IDF LinearSVC — same full-tree run as hierarchical_1b (not refit here).
    """
    pkl = base / "full_tree_rows.pkl"
    if not pkl.is_file():
        raise FileNotFoundError(
            f"Missing {pkl}. Run the full-tree training cell in hierarchical_1b.ipynb first."
        )
    with open(pkl, "rb") as f:
        full_tree_rows = pickle.load(f)
    for r in full_tree_rows:
        if r.get("model") == "LinearSVC":
            out = {k: v for k, v in r.items() if not isinstance(v, (dict, list))}
            out["model"] = "SVM_C_1.0_from_1b"
            out["C"] = 1.0
            out["source"] = "hierarchical_1b full_tree_rows.pkl (LinearSVC, default C=1)"
            return out
    raise KeyError('No row with model=="LinearSVC" in full_tree_rows.pkl')


BASE = homework4_base()
tree = load_topic_tree(str(BASE / "topics.csv"))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
SPLIT = "test"
RESTRICT = True
MAX_FEATURES = 8000

print("BASE", BASE)
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

rows_c_trained: list[dict] = []
router_by_depth = None
row_by_depth = None
by_d_depth = None


BASE /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4
train 14465 test 3617


### Training — global `C` grid (coarse)

Full refit **for each** `C` in **`C_GRID`** — **omit `1.0`**: default slack **`C=1`** is taken from **`full_tree_rows.pkl`** (see results), not trained again. **`verbose=True`**. By-depth F1 per trained `C` → `problem3_phase3_by_depth_C_<C>.json`.

Set **`RUN_PHASE3 = True`** to run (slow).


In [3]:
RUN_PHASE3 = False
# Omit C=1.0 — that slack setting is the 1b LinearSVC row (merged in results), not refit here.
C_GRID = [1e-2, 0.1, 10.0]

if RUN_PHASE3:
    rows_c_trained = []
    for mi, C in enumerate(C_GRID, start=1):
        print("\n" + "=" * 72, flush=True)
        print(f"GLOBAL C [{mi}/{len(C_GRID)}] C = {C}", flush=True)
        print("=" * 72 + "\n", flush=True)
        fac = svm_linear_binary_edge_factory(
            max_features=MAX_FEATURES,
            text_vectorizer="tfidf",
            svm_kw=dict(C=float(C), dual=False, max_iter=8000),
        )
        t0 = time.perf_counter()
        router, row = train_full_tree_and_summarize(
            f"SVM_C_{C}",
            tree,
            mldata,
            fac,
            split=SPLIT,
            verbose=True,
            restrict_to_parent_subtree=RESTRICT,
        )
        row["wall_time_sec"] = time.perf_counter() - t0
        row["C"] = C
        by_d = pooled_edge_f1_stats_by_parent_depth(
            router, mldata, tree, SPLIT, restrict_to_parent_subtree=RESTRICT
        )
        row_flat = {k: v for k, v in row.items() if not isinstance(v, (dict, list))}
        rows_c_trained.append(row_flat)
        with open(BASE / f"problem3_phase3_by_depth_C_{C}.json", "w") as f:
            json.dump({str(d): by_d[d]["pooled_micro_f1"] for d in by_d}, f, indent=2)
        print(
            f"\n>>> SVM_C_{C} finished in {row['wall_time_sec']:.1f}s wall time\n",
            flush=True,
        )
else:
    print("SKIP global C training (RUN_PHASE3=False).")



SKIP global C training (RUN_PHASE3=False).


### Results — global `C` grid

Merges **`C=1`** (default slack) from **`row_linear_svc_c1_from_1b`** with **`rows_c_trained`**, sorted by **`C`**, then writes **`problem3_phase3_global_C.csv`**. Set **`INCLUDE_LINEAR_SVC_C1_FROM_1B = False`** if the pickle is unavailable and you only want trained rows.

In [4]:
INCLUDE_LINEAR_SVC_C1_FROM_1B = True

rows_global: list[dict] = []
if INCLUDE_LINEAR_SVC_C1_FROM_1B:
    try:
        rows_global.append(row_linear_svc_c1_from_1b(BASE))
    except (FileNotFoundError, KeyError) as e:
        print("C=1 (default slack) from 1b:", e)
rows_global.extend(rows_c_trained)

if rows_global:
    rows_global = sorted(rows_global, key=lambda r: float(r.get("C", 0)))
    df_c = comparison_table(rows_global)
    cols = ["model", "C", "ft_pooled_micro_f1", "leaf_macro_f1", "wall_time_sec"]
    cols = [c for c in cols if c in df_c.columns]
    display(df_c[cols].round(4))
    pd.DataFrame(rows_global).to_csv(BASE / "problem3_phase3_global_C.csv", index=False)
    print("Wrote", BASE / "problem3_phase3_global_C.csv")
else:
    df_c = None
    print("No rows: enable INCLUDE_LINEAR_SVC_C1_FROM_1B and/or set RUN_PHASE3=True.")


No global C rows (set RUN_PHASE3=True and re-run training).


### Training — per-depth `C` map (optional)

**`LinearSVC`** with slack **`C`** set from parent depth via [`linear_svc_estimator_by_depth`](hierarchical_classifier.py). **`DEFAULT_C`** is the slack for any depth **not** listed in **`C_BY_DEPTH`**. Set **`RUN_DEPTH_MAP = True`** after choosing the map from the global-`C` table. **`verbose=True`** here.

**Next section:** by-depth table and aggregate metrics.


In [5]:
RUN_DEPTH_MAP = False
# Slack C per parent depth; DEFAULT_C applies to depths not in the dict
C_BY_DEPTH = {0: 1.0, 1: 1.0, 2: 0.5, 3: 0.5, 4: 0.5}
DEFAULT_C = 1.0

if RUN_DEPTH_MAP:
    print("\n" + "=" * 72, flush=True)
    print("PER-DEPTH C MAP — training", flush=True)
    print("=" * 72 + "\n", flush=True)
    est = linear_svc_estimator_by_depth(C_BY_DEPTH, default_c=DEFAULT_C)
    fac = binary_edge_factory(
        tfidf_kw=dict(min_df=2, max_features=MAX_FEATURES),
        estimator=est,
        clf_kw=None,
        text_vectorizer="tfidf",
    )
    t0 = time.perf_counter()
    router_by_depth, row_by_depth = train_full_tree_and_summarize(
        "SVM_C_by_depth",
        tree,
        mldata,
        fac,
        split=SPLIT,
        verbose=True,
        restrict_to_parent_subtree=RESTRICT,
    )
    row_by_depth["wall_time_sec"] = time.perf_counter() - t0
    row_by_depth["C_by_depth"] = str(C_BY_DEPTH)
    by_d_depth = pooled_edge_f1_stats_by_parent_depth(
        router_by_depth, mldata, tree, SPLIT, restrict_to_parent_subtree=RESTRICT
    )
    with open(BASE / "problem3_phase3_by_depth_map.json", "w") as f:
        json.dump(
            {str(d): by_d_depth[d]["pooled_micro_f1"] for d in by_d_depth},
            f,
            indent=2,
        )
    print(
        f"\n>>> SVM_C_by_depth finished in {row_by_depth['wall_time_sec']:.1f}s wall time\n",
        flush=True,
    )
else:
    print("SKIP per-depth map training (RUN_DEPTH_MAP=False).")


SKIP per-depth map training (RUN_DEPTH_MAP=False).


### Results — per-depth `C` map

Runs if **`row_by_depth`** was set in the previous cell. Shows the by-depth table, **`ft_pooled_micro_f1`**, and writes **`problem3_phase3_by_depth_map.csv`** (one row of flattened metrics).


In [ ]:
if row_by_depth is not None and by_d_depth is not None:
    display(
        pd.DataFrame(
            {
                d: {
                    "pooled_micro_f1": by_d_depth[d]["pooled_micro_f1"],
                    "n_edges": by_d_depth[d]["n_edges_used"],
                }
                for d in by_d_depth
            }
        ).T
    )
    print("ft_pooled_micro_f1", row_by_depth.get("ft_pooled_micro_f1"))
    flat = {
        k: v
        for k, v in row_by_depth.items()
        if not isinstance(v, (dict, list))
    }
    pd.DataFrame([flat]).to_csv(BASE / "problem3_phase3_by_depth_map.csv", index=False)
    print("Wrote", BASE / "problem3_phase3_by_depth_map.csv")
else:
    print("No per-depth run (set RUN_DEPTH_MAP=True and re-run training).")


### Optional: by-depth metrics for any `router`

If you loaded a saved router elsewhere:


In [ ]:
# Example: last trained per-depth router in this session
# if router_by_depth is not None:
#     by_d = pooled_edge_f1_stats_by_parent_depth(
#         router_by_depth, mldata, tree, SPLIT, restrict_to_parent_subtree=RESTRICT
#     )
#     pd.DataFrame({d: by_d[d] for d in sorted(by_d)}).T
